# Interpretable Predictions for Steam Game Recommendations

**Author:** Michael Theophanopoulos  
**Purpose:** Binary classifier with counterfactual explanations  
**Last Updated:** 2025-10-29

---

## Notebook Structure

1. Data Loading & Exploration
2. Feature Engineering
3. Model Training
4. Prediction Function (with Counterfactuals)
5. Model Evaluation

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
from pathlib import Path
import logging

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Simple logging setup
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

logger.info("Notebook initialized successfully")

---

## 0. Data Scraping

Scape the data from https://store.steampowered.com/appreviews/ and store them in a csv file.

In [88]:
import requests
import pandas as pd
import time
from pathlib import Path
from typing import List, Dict, Optional
import logging

In [89]:
def scrape_steam_reviews(
    app_id: int,
    max_reviews: int = 50000,
    language: str = 'english',
    reviews_per_page: int = 100
) -> List[Dict]:
    """
    Scrape game reviews from Steam API.
    
    Parameters
    ----------
    app_id : int
        Steam application ID (default: 1245620 for Elden Ring)
    max_reviews : int
        Maximum number of reviews to collect
    language : str
        Review language filter
    reviews_per_page : int
        Number of reviews per API request (max 100)
        
    Returns
    -------
    List[Dict]
        List of review dictionaries
    """
    all_reviews = []
    cursor = '*'
    start_time = time.time()
    
    logging.info(f"Starting fetching reviews for app_id={app_id}...")
    
    while len(all_reviews) < max_reviews:
        url = f'https://store.steampowered.com/appreviews/{app_id}'
        params = {
            'json': 1,
            'language': language,
            'cursor': cursor,
            'num_per_page': reviews_per_page,
            'filter': 'recent'
        }
        
        try:
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            
            if not data.get('reviews'):
                logging.warning("No more reviews available")
                break
            
            all_reviews.extend(data['reviews'])
            cursor = data.get('cursor')
            
            if not cursor:
                logging.warning("No cursor returned, ending pagination")
                break
            
            # Rate limiting: 0.3s between requests
            time.sleep(0.3)
            
        except requests.exceptions.RequestException as e:
            logging.error(f"Request failed: {e}")
            time.sleep(5)
    
    final_reviews = all_reviews[:max_reviews]
    elapsed_time = time.time() - start_time
    
    logging.info(f"Fetched {len(final_reviews)} reviews in {elapsed_time:.2f} seconds")
    
    return final_reviews

In [90]:
def fetch_game_details(app_id: int) -> Dict:
    """
    Fetch game metadata from Steam Store API.
    
    Parameters
    ----------
    app_id : int
        Steam application ID
        
    Returns
    -------
    Dict
        Game metadata including price, genres, description, etc.
    """
    url = f'https://store.steampowered.com/api/appdetails'
    params = {'appids': app_id}
    
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if str(app_id) in data and data[str(app_id)]['success']:
            game_data = data[str(app_id)]['data']
            logging.info(f"Fetched game details for '{game_data.get('name', 'Unknown')}'")
            return game_data
        else:
            logging.warning(f"Failed to fetch game details for app_id={app_id}")
            return {}
            
    except requests.exceptions.RequestException as e:
        logging.error(f"Request failed: {e}")
        return {}


def process_reviews(reviews: List[Dict], game_data: Dict = None) -> pd.DataFrame:
    """
    Convert raw review data to structured DataFrame.
    
    Parameters
    ----------
    reviews : List[Dict]
        Raw review data from Steam API
    game_data : Dict, optional
        Game metadata from appdetails API (price, genre, description, etc.)
        
    Returns
    -------
    pd.DataFrame
        Structured dataset with selected features for binary classification
    """
    data = []
    
    for review in reviews:
        author = review.get('author', {})
        
        record = {
            # Target variable
            'recommended': review.get('voted_up', False),
            
            # User attributes (descriptive features)
            'user_id': author.get('steamid', ''),
            'user_games_owned': author.get('num_games_owned', 0),
            'user_num_reviews': author.get('num_reviews', 0),
            
            # User-Game interaction (numeric features)
            'playtime_at_review_hours': author.get('playtime_at_review', 0) / 60,
            'playtime_total_hours': author.get('playtime_forever', 0) / 60,
            'playtime_recent_hours': author.get('playtime_last_two_weeks', 0) / 60,
            
            # Review metadata (categorical features)
            'received_free': review.get('received_for_free', False),
            'steam_purchase': review.get('steam_purchase', True),
            'written_early_access': review.get('written_during_early_access', False),
            
            # Review engagement (numeric features)
            'votes_helpful': review.get('votes_up', 0),
            'votes_funny': review.get('votes_funny', 0),
            'weighted_vote_score': review.get('weighted_vote_score', 0.0),
            'comment_count': review.get('comment_count', 0),
            
            # Temporal features
            'timestamp_created': review.get('timestamp_created', 0),
            'timestamp_updated': review.get('timestamp_updated', 0),
            
            # Text features (free text)
            'review_text': review.get('review', ''),
            'review_length': len(review.get('review', '')),
            
            # Unique identifiers
            'review_id': review.get('recommendationid', '')
        }
        
        # Add game features if game_data provided
        if game_data:
            record.update({
                'game_id': game_data.get('steam_appid', ''),
                'game_name': game_data.get('name', ''),
                'game_price': game_data.get('price_overview', {}).get('final', 0) / 100 if game_data.get('price_overview') else 0,
                'game_is_free': game_data.get('is_free', False),
                'game_genre': ','.join([g['description'] for g in game_data.get('genres', [])]),
                'game_categories': ','.join([c['description'] for c in game_data.get('categories', [])]),
                'game_developer': ','.join(game_data.get('developers', [])),
                'game_publisher': ','.join(game_data.get('publishers', [])),
                'game_description': game_data.get('short_description', ''),
                'game_required_age': game_data.get('required_age', 0),
            })
        
        data.append(record)
    
    df = pd.DataFrame(data)
    
    # Feature engineering
    df['playtime_ratio'] = df['playtime_at_review_hours'] / (df['playtime_total_hours'] + 1)
    df['review_engagement_score'] = df['votes_helpful'] + df['votes_funny'] * 0.5
    df['experienced_gamer'] = df['user_games_owned'] > 50
    df['active_reviewer'] = df['user_num_reviews'] > 5
    
    feature_count = len(df.columns)
    logging.info(f"Processed {len(df)} reviews into DataFrame with {feature_count} features")
    
    return df

In [91]:
# Configuration
APP_ID = 578080  # PUBG
MAX_REVIEWS = 50000
OUTPUT_FILE = 'steam_reviews.csv'

# Fetch game metadata first
logging.info(f"Fetching game metadata for app_id={APP_ID}...")
game_data = fetch_game_details(APP_ID)

# Collect reviews
logging.info(f"Starting to scrape reviews for app_id={APP_ID}...")
reviews = scrape_steam_reviews(app_id=APP_ID, max_reviews=MAX_REVIEWS)

INFO:root:Fetching game metadata for app_id=578080...
INFO:root:Fetched game details for 'PUBG: BATTLEGROUNDS'
INFO:root:Starting to scrape reviews for app_id=578080...
INFO:root:Starting fetching reviews for app_id=578080...
INFO:root:Fetched 22700 reviews in 180.59 seconds


In [92]:
# Process reviews with game data and save to CSV
df = process_reviews(reviews, game_data=game_data)
df.to_csv(OUTPUT_FILE, index=False)
logging.info(f"Dataset saved to {OUTPUT_FILE}")

INFO:root:Processed 22700 reviews into DataFrame with 33 features
INFO:root:Dataset saved to steam_reviews.csv


In [93]:
# Data summary
print(f"Dataset shape: {df.shape}")
print(f"Positive reviews: {df['recommended'].sum()} ({df['recommended'].mean()*100:.1f}%)")

# Show game info if available
if 'game_name' in df.columns:
    print(f"\nGame: {df['game_name'].iloc[0]}")
    print(f"Price: ${df['game_price'].iloc[0]:.2f}")
    print(f"Genre: {df['game_genre'].iloc[0]}")
    print(f"Description: {df['game_description'].iloc[0][:100]}...")

print(f"\nFirst few rows:")
df.head()

Dataset shape: (22700, 33)
Positive reviews: 15628 (68.8%)

Game: PUBG: BATTLEGROUNDS
Price: $0.00
Genre: Action,Adventure,Massively Multiplayer,Free To Play
Description: PUBG: BATTLEGROUNDS, the high-stakes winner-take-all shooter that started the Battle Royale craze, i...

First few rows:


,recommended,user_id,user_games_owned,user_num_reviews,playtime_at_review_hours,playtime_total_hours,playtime_recent_hours,received_free,steam_purchase,written_early_access,...,game_genre,game_categories,game_developer,game_publisher,game_description,game_required_age,playtime_ratio,review_engagement_score,experienced_gamer,active_reviewer
0,True,76561197990088609,0,11,80.316667,80.316667,0.000000,False,True,False,...,"Action,Adventure,Massively Multiplayer,Free To...","Multi-player,PvP,Online PvP,Stats,Remote Play ...",PUBG Corporation,"KRAFTON, Inc.","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",0,0.987702,0.0,False,True
1,True,76561198347370385,78,2,351.566667,352.650000,25.266667,False,True,False,...,"Action,Adventure,Massively Multiplayer,Free To...","Multi-player,PvP,Online PvP,Stats,Remote Play ...",PUBG Corporation,"KRAFTON, Inc.","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",0,0.994109,0.0,True,False
2,False,76561198372979773,281,26,13.900000,13.900000,1.583333,False,True,False,...,"Action,Adventure,Massively Multiplayer,Free To...","Multi-player,PvP,Online PvP,Stats,Remote Play ...",PUBG Corporation,"KRAFTON, Inc.","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",0,0.932886,4.5,True,True
3,False,76561198027390661,189,13,168.216667,168.216667,0.000000,False,True,False,...,"Action,Adventure,Massively Multiplayer,Free To...","Multi-player,PvP,Online PvP,Stats,Remote Play ...",PUBG Corporation,"KRAFTON, Inc.","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",0,0.994090,0.0,True,True
4,True,76561198310415102,61,6,218.083333,218.716667,7.516667,False,True,False,...,"Action,Adventure,Massively Multiplayer,Free To...","Multi-player,PvP,Online PvP,Stats,Remote Play ...",PUBG Corporation,"KRAFTON, Inc.","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",0,0.992566,0.0,True,True


---

## 1. Data Loading & Exploration

Load the preprocessed Steam reviews dataset and perform exploratory analysis.

In [15]:
from pathlib import Path


In [16]:
# File paths
DATA_FILE = Path('steam_reviews.csv')
# Validate file exists
if not DATA_FILE.exists():
    logger.error(f"Data file not found: {DATA_FILE}")
    raise FileNotFoundError(
        f"Data file not found: {DATA_FILE}. "
        "Please run data collection first."
    )

# Load data
logger.info(f"Loading data from {DATA_FILE}")
df = pd.read_csv(DATA_FILE)
logger.info(f"Loaded {len(df):,} reviews with {df.shape[1]} columns")

# Display shape
print(f"\nDataset shape: {df.shape}")
df.head()

INFO     | Loading data from steam_reviews.csv
INFO     | Loaded 22,700 reviews with 33 columns

Dataset shape: (22700, 33)


,recommended,user_id,user_games_owned,user_num_reviews,playtime_at_review_hours,playtime_total_hours,playtime_recent_hours,received_free,steam_purchase,written_early_access,...,game_genre,game_categories,game_developer,game_publisher,game_description,game_required_age,playtime_ratio,review_engagement_score,experienced_gamer,active_reviewer
0,True,76561197990088609,0,11,80.316667,80.316667,0.000000,False,True,False,...,"Action,Adventure,Massively Multiplayer,Free To...","Multi-player,PvP,Online PvP,Stats,Remote Play ...",PUBG Corporation,"KRAFTON, Inc.","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",0,0.987702,0.0,False,True
1,True,76561198347370385,78,2,351.566667,352.650000,25.266667,False,True,False,...,"Action,Adventure,Massively Multiplayer,Free To...","Multi-player,PvP,Online PvP,Stats,Remote Play ...",PUBG Corporation,"KRAFTON, Inc.","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",0,0.994109,0.0,True,False
2,False,76561198372979773,281,26,13.900000,13.900000,1.583333,False,True,False,...,"Action,Adventure,Massively Multiplayer,Free To...","Multi-player,PvP,Online PvP,Stats,Remote Play ...",PUBG Corporation,"KRAFTON, Inc.","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",0,0.932886,4.5,True,True
3,False,76561198027390661,189,13,168.216667,168.216667,0.000000,False,True,False,...,"Action,Adventure,Massively Multiplayer,Free To...","Multi-player,PvP,Online PvP,Stats,Remote Play ...",PUBG Corporation,"KRAFTON, Inc.","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",0,0.994090,0.0,True,True
4,True,76561198310415102,61,6,218.083333,218.716667,7.516667,False,True,False,...,"Action,Adventure,Massively Multiplayer,Free To...","Multi-player,PvP,Online PvP,Stats,Remote Play ...",PUBG Corporation,"KRAFTON, Inc.","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",0,0.992566,0.0,True,True


In [17]:
# Dataset overview
print("="*60)
print("DATASET SUMMARY")
print("="*60)
print(f"Total records: {len(df):,}")
print(f"Features: {df.shape[1]}")
print(f"\nTarget distribution:")
print(f"  Positive reviews: {df['recommended'].sum():,} ({df['recommended'].mean()*100:.2f}%)")
print(f"  Negative reviews: {(~df['recommended']).sum():,} ({(~df['recommended']).mean()*100:.2f}%)")
print("\n" + "="*60)

DATASET SUMMARY
Total records: 22,700
Features: 33

Target distribution:
  Positive reviews: 15,628 (68.85%)
  Negative reviews: 7,072 (31.15%)



In [18]:
# Data quality check
print("DATA QUALITY REPORT")
print("="*60)
print("\nMissing values:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)
print("\nBasic statistics:")
df.describe()

DATA QUALITY REPORT

Missing values:
recommended                  0
user_id                      0
user_games_owned             0
user_num_reviews             0
playtime_at_review_hours     0
playtime_total_hours         0
playtime_recent_hours        0
received_free                0
steam_purchase               0
written_early_access         0
votes_helpful                0
votes_funny                  0
weighted_vote_score          0
comment_count                0
timestamp_created            0
timestamp_updated            0
review_text                 90
review_length                0
review_id                    0
game_id                      0
game_name                    0
game_price                   0
game_is_free                 0
game_genre                   0
game_categories              0
game_developer               0
game_publisher               0
game_description             0
game_required_age            0
playtime_ratio               0
review_engagement_score      0
ex

,user_id,user_games_owned,user_num_reviews,playtime_at_review_hours,playtime_total_hours,playtime_recent_hours,votes_helpful,votes_funny,weighted_vote_score,comment_count,timestamp_created,timestamp_updated,review_length,review_id,game_id,game_price,game_required_age,playtime_ratio,review_engagement_score
count,2.270000e+04,22700.000000,22700.000000,22700.000000,22700.000000,22700.000000,22700.000000,22700.000000,22700.000000,22700.000000,2.270000e+04,2.270000e+04,22700.000000,2.270000e+04,22700.0,22700.0,22700.0,22700.000000,22700.000000
mean,7.656120e+16,63.317930,9.889427,776.906423,1014.402471,4.111514,1.052159,0.186079,0.503354,0.044714,1.703303e+09,1.704706e+09,99.694934,1.555447e+08,578080.0,0.0,0.0,0.768036,1.145198
std,3.752984e+08,303.814404,49.174921,1287.777803,1597.093465,13.684895,16.009169,2.696296,0.025123,0.487638,2.976462e+07,3.002258e+07,275.386239,2.567061e+07,0.0,0.0,0.0,0.246094,16.818483
min,7.656120e+16,0.000000,1.000000,0.116667,0.150000,0.000000,0.000000,0.000000,0.317252,0.000000,1.656630e+09,1.656630e+09,0.000000,1.179596e+08,578080.0,0.0,0.0,0.000254,0.000000
25%,7.656120e+16,0.000000,1.000000,81.562500,128.987500,0.000000,0.000000,0.000000,0.500000,0.000000,1.676966e+09,1.678327e+09,8.000000,1.332806e+08,578080.0,0.0,0.0,0.641941,0.000000
50%,7.656120e+16,0.000000,3.000000,316.366667,435.208333,0.000000,0.000000,0.000000,0.500000,0.000000,1.702223e+09,1.703773e+09,24.000000,1.533067e+08,578080.0,0.0,0.0,0.861257,0.000000
75%,7.656120e+16,50.000000,8.000000,933.825000,1205.954167,0.000000,0.000000,0.000000,0.500000,0.000000,1.727378e+09,1.729189e+09,80.000000,1.758303e+08,578080.0,0.0,0.0,0.965894,0.000000
max,7.656120e+16,28185.000000,6066.000000,28506.916667,30239.783333,297.600000,1636.000000,145.000000,0.938077,23.000000,1.761706e+09,1.761706e+09,7832.000000,2.078426e+08,578080.0,0.0,0.0,0.999898,1671.500000


---

## 2. Feature Engineering

Prepare features from review text, playtime, and other variables for the binary classifier.

In [19]:
# Feature engineering imports
from textblob import TextBlob

In [ ]:
# Feature Engineering
logger.info("Starting feature engineering")

# Handle missing review texts
df['review_text'] = df['review_text'].fillna('')

# Extract text features
from textblob import TextBlob

df['word_count'] = df['review_text'].apply(lambda x: len(str(x).split()))
df['sentiment_polarity'] = df['review_text'].apply(
    lambda x: TextBlob(str(x)).sentiment.polarity
)

# Log transform playtime
df['log_playtime_at_review'] = np.log1p(df['playtime_at_review_hours'])

# Handle missing values
df['game_price'] = df['game_price'].fillna(0)
df['game_genre'] = df['game_genre'].fillna('Unknown')
df['game_description'] = df['game_description'].fillna('')

# Select features for modeling
features = [
    'recommended',  # Target
    'user_games_owned',
    'user_num_reviews',
    'log_playtime_at_review',
    'game_price',
    'game_genre',
    'game_description',
    'review_text',
    'sentiment_polarity',
    'word_count'
]

df_subset = df[features].copy()

logger.info(f"Feature engineering complete. Shape: {df_subset.shape}")
print(f"\nDataset shape: {df_subset.shape}")
df_subset.head()

---

## 3. Model Training

Train the binary classifier to predict whether a user will recommend a game.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [ ]:
def train_classifier(X_train, y_train, model=None):
    """
    Train a binary classifier with preprocessing pipeline.
    
    Args:
        X_train: Training features
        y_train: Training labels
        model: Sklearn classifier (default: LogisticRegression)
    
    Returns:
        Trained pipeline
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import LogisticRegression
    
    # Define feature types
    numeric = ['user_games_owned', 'game_price', 'user_num_reviews', 
               'log_playtime_at_review', 'sentiment_polarity', 'word_count']
    categorical = ['game_genre']
    
    # Build preprocessor
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), numeric),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
        ('review', TfidfVectorizer(max_features=100, stop_words='english'), 'review_text'),
        ('desc', TfidfVectorizer(max_features=50, stop_words='english'), 'game_description')
    ])
    
    # Create pipeline
    if model is None:
        model = LogisticRegression(random_state=42, max_iter=1000)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Train
    logger.info(f"Training {model.__class__.__name__} on {len(X_train):,} samples")
    pipeline.fit(X_train, y_train)
    logger.info("Training complete")
    
    return pipeline

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

logger.info("="*60)
logger.info("Starting Model Training Phase")
logger.info("="*60)

# Prepare data once
logger.info("Preparing train-test split")
X = df_subset[features_keep[1:]]  # Exclude target
y = df_subset['recommended']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

logger.info(f"Training set: {len(X_train):,} samples")
logger.info(f"Test set: {len(X_test):,} samples")
logger.info(f"Class distribution (train): Positive={y_train.sum():,} ({y_train.mean()*100:.2f}%), Negative={(~y_train).sum():,} ({(~y_train).mean()*100:.2f}%)")

# Train different models (excluding GaussianNB - incompatible with sparse matrices from TF-IDF)
logger.info("Training multiple model variants...")

logger.info("1. Training Logistic Regression...")
pipeline_lr = train_classifier(X_train, y_train, 
                               model=LogisticRegression(random_state=42, max_iter=1000))

logger.info("2. Training Random Forest...")
pipeline_rf = train_classifier(X_train, y_train,
                               model=RandomForestClassifier(n_estimators=100, random_state=42))

logger.info("3. Training XGBoost...")
pipeline_xgb = train_classifier(X_train, y_train,
                                model=XGBClassifier(random_state=42, eval_metric='logloss'))

logger.info("4. Training SVM...")
pipeline_svm = train_classifier(X_train, y_train,
                                model=SVC(probability=True, random_state=42))

logger.info("5. Training default model...")
pipeline_default = train_classifier(X_train, y_train)

logger.info("="*60)
logger.info("All models trained successfully")
logger.info("="*60)

INFO     | ============================================================
INFO     | Starting Model Training Phase
INFO     | ============================================================
INFO     | Preparing train-test split
INFO     | Training set: 18,160 samples
INFO     | Test set: 4,540 samples
INFO     | Class distribution (train): Positive=12,477 (68.71%), Negative=5,683 (31.29%)
INFO     | Training multiple model variants...
INFO     | 1. Training Logistic Regression...
INFO     | Initializing classifier training pipeline
INFO     | Numeric features: 6
INFO     | Categorical features: 1
INFO     | Text features: 2
INFO     | Building preprocessing pipeline
INFO     | Using provided model: LogisticRegression
INFO     | Training on 18,160 samples with 9 features
INFO     | Model training completed successfully
INFO     | 2. Training Random Forest...
INFO     | Initializing classifier training pipeline
INFO     | Numeric features: 6
INFO     | Categorical features: 1
INFO     | Text 

In [24]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import time

# Define models to compare (excluding GaussianNB - incompatible with sparse matrices)
models_to_test = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'SVM': SVC(probability=True, random_state=42)
}

logger.info(f"Comparing {len(models_to_test)} models")

# Compare all models
results = []

for model_name, model in models_to_test.items():
    print(f"\n{'='*60}")
    print(f"Training {model_name}...")
    print('='*60)
    
    # Train
    start_time = time.time()
    pipeline = train_classifier(X_train, y_train, model=model)
    training_time = time.time() - start_time
    
    # Predict
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    # Evaluate
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    logger.info(f"{model_name}: Accuracy={accuracy:.4f}, AUC-ROC={auc:.4f}, Time={training_time:.2f}s")
    
    results.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'AUC-ROC': auc,
        'Training Time (s)': training_time
    })
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"AUC-ROC: {auc:.4f}")

# Summary
results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)
print(results_df.to_string(index=False))

# Get best model
best_model_name = results_df.iloc[0]['Model']
print(f"\n Best Model: {best_model_name}")
print(f"   Accuracy: {results_df.iloc[0]['Accuracy']:.4f}")

logger.info(f"Best model: {best_model_name} with accuracy {results_df.iloc[0]['Accuracy']:.4f}")

INFO     | Comparing 5 models

Training Logistic Regression...
INFO     | Initializing classifier training pipeline
INFO     | Numeric features: 6
INFO     | Categorical features: 1
INFO     | Text features: 2
INFO     | Building preprocessing pipeline
INFO     | Using provided model: LogisticRegression
INFO     | Training on 18,160 samples with 9 features
INFO     | Model training completed successfully
INFO     | Logistic Regression: Accuracy=0.8192, AUC-ROC=0.8667, Time=0.33s
Accuracy: 0.8192
AUC-ROC: 0.8667

Training Random Forest...
INFO     | Initializing classifier training pipeline
INFO     | Numeric features: 6
INFO     | Categorical features: 1
INFO     | Text features: 2
INFO     | Building preprocessing pipeline
INFO     | Using provided model: RandomForestClassifier
INFO     | Training on 18,160 samples with 9 features
INFO     | Model training completed successfully
INFO     | Random Forest: Accuracy=0.8104, AUC-ROC=0.8660, Time=4.85s
Accuracy: 0.8104
AUC-ROC: 0.8660

Tra

---

## 4. Prediction Function (with Counterfactuals)

Create a function that:
- Takes a user + game input
- Outputs binary prediction (yes/no they'll like it)
- If prediction is negative: generates counterfactual explanations

In [50]:
def predict_with_counterfactuals(model, user_features, threshold=0.5):
    """
    Predict and generate counterfactuals.

    Parameters
    ----------
    model : trained model
        Trained sklearn pipeline
    user_features : pd.DataFrame
        Feature dataframe with all required columns
    threshold : float, optional
        Classification threshold (default: 0.5)

    Returns
    -------
    dict with prediction, probability, counterfactuals
    """

    # Predict
    prediction = model.predict(user_features)[0]
    prob = model.predict_proba(user_features)[0][1]

    # Counterfactuals if negative prediction
    counterfactuals = []
    if not prediction:  # If prediction is False (not recommended)
        feature_names = user_features.columns.tolist()
        counterfactuals = generate_counterfactuals(model, user_features, feature_names)

    return {
        'prediction': bool(prediction),
        'probability': float(prob),
        'counterfactuals': counterfactuals
    }

In [51]:
def generate_counterfactuals(model, user_features, feature_names):
    """
    Generate counterfactuals using simple greedy search.
    
    Strategy: Try increasing each numeric feature until prediction flips.
    
    Args:
        model: Trained classifier
        user_features: DataFrame with user's features (1 row)
        feature_names: List of feature names
    
    Returns:
        List of counterfactual suggestions
    """
    counterfactuals = []
    
    # Features we can modify
    modifiable = ['user_num_reviews', 'log_playtime_at_review', 
                  'sentiment_polarity', 'word_count']
    
    for feature in modifiable:
        if feature not in feature_names:
            continue
        
        modified = user_features.copy()
        original_val = modified[feature].values[0]
        
        # Define test values
        if feature == 'sentiment_polarity':
            test_values = [0.2, 0.4, 0.6, 0.8]
        elif feature == 'log_playtime_at_review':
            test_values = [original_val + i for i in [0.5, 1.0, 1.5, 2.0]]
        elif feature == 'word_count':
            test_values = [original_val + i for i in [10, 20, 50, 100]]
        else:  # user_num_reviews
            test_values = [original_val + i for i in [5, 10, 20, 50]]
        
        # Test each value
        for test_val in test_values:
            modified[feature] = test_val
            
            try:
                if model.predict(modified)[0]:  # Prediction flipped to positive
                    # Create explanation
                    if feature == 'log_playtime_at_review':
                        old_hours = np.expm1(original_val)
                        new_hours = np.expm1(test_val)
                        explanation = f"Increase playtime from {old_hours:.1f} to {new_hours:.1f} hours"
                    elif feature == 'sentiment_polarity':
                        explanation = f"Improve sentiment from {original_val:.2f} to {test_val:.2f}"
                    elif feature == 'word_count':
                        explanation = f"Write longer review ({int(original_val)} to {int(test_val)} words)"
                    else:
                        explanation = f"Write more reviews ({int(original_val)} to {int(test_val)})"
                    
                    counterfactuals.append({
                        'feature': feature,
                        'original': float(original_val),
                        'suggested': float(test_val),
                        'change': explanation,
                        'method': 'greedy'
                    })
                    break
            except:
                continue
    
    return counterfactuals

In [52]:
def generate_counterfactuals_dice(model, user_features, feature_names, num_cfs=3):
    """
    Generate counterfactuals using DICE-inspired approach.
    
    Strategy: Generate diverse counterfactuals by changing multiple features.
    
    Args:
        model: Trained classifier
        user_features: DataFrame with user's features (1 row)
        feature_names: List of feature names
        num_cfs: Number of counterfactuals to generate
    
    Returns:
        List of counterfactual explanations
    """
    logger.info("Generating DICE counterfactuals")
    
    counterfactuals = []
    
    # Numeric features we can modify
    numeric = ['user_games_owned', 'user_num_reviews', 
               'log_playtime_at_review', 'sentiment_polarity', 'word_count']
    available = [f for f in numeric if f in feature_names]
    
    # Different modification strategies (diverse solutions)
    strategies = [
        {'sentiment_polarity': 0.3, 'word_count': 20},  # Better review
        {'log_playtime_at_review': 1.5, 'sentiment_polarity': 0.2},  # More playtime + sentiment
        {'user_num_reviews': 10, 'log_playtime_at_review': 1.0},  # More engagement
    ]
    
    for i, strategy in enumerate(strategies[:num_cfs]):
        modified = user_features.copy()
        changes = {}
        
        # Apply changes
        for feature, delta in strategy.items():
            if feature not in available:
                continue
            
            old_val = modified[feature].values[0]
            
            # Update value
            if feature == 'sentiment_polarity':
                new_val = min(1.0, max(-1.0, old_val + delta))
            else:
                new_val = old_val + delta
            
            modified[feature] = new_val
            changes[feature] = (float(old_val), float(new_val))
        
        # Check if prediction flips
        try:
            if model.predict(modified)[0]:  # Flipped to positive
                # Calculate distance
                distance = np.sqrt(sum((new - old)**2 for old, new in changes.values()))
                
                # Build explanation
                descriptions = []
                for feature, (old_val, new_val) in changes.items():
                    if feature == 'log_playtime_at_review':
                        descriptions.append(
                            f"playtime: {np.expm1(old_val):.0f}h → {np.expm1(new_val):.0f}h"
                        )
                    elif feature == 'sentiment_polarity':
                        descriptions.append(f"sentiment: {old_val:.2f} → {new_val:.2f}")
                    elif feature == 'word_count':
                        descriptions.append(f"review: {int(old_val)} → {int(new_val)} words")
                    else:
                        descriptions.append(f"{feature}: {old_val:.0f} → {new_val:.0f}")
                
                counterfactuals.append({
                    'explanation': ", ".join(descriptions),
                    'feature_changes': changes,
                    'distance': float(distance),
                    'method': 'dice'
                })
        except:
            continue
    
    logger.info(f"Generated {len(counterfactuals)} counterfactuals")
    return counterfactuals

In [53]:
def generate_anchors_explanation(model, user_features, feature_names, threshold=0.85):
    """
    Generate rule-based explanation using Anchors approach.
    
    Strategy: Find which features are most critical for the prediction.
    
    Args:
        model: Trained classifier
        user_features: DataFrame with user's features (1 row)
        feature_names: List of feature names
        threshold: Minimum precision for anchor rule
    
    Returns:
        Dictionary with anchor rule and metrics
    """
    logger.info("Generating anchor explanation")
    
    # Numeric features to analyze
    numeric = ['user_games_owned', 'user_num_reviews', 
               'log_playtime_at_review', 'sentiment_polarity', 'word_count']
    available = [f for f in numeric if f in feature_names]
    
    # Get original prediction
    original_pred = model.predict(user_features)[0]
    
    # Test feature importance by perturbation
    feature_stability = []
    
    for feature in available:
        predictions = []
        
        # Generate 50 test instances
        for _ in range(50):
            test = user_features.copy()
            
            # Perturb all features EXCEPT the one we're testing
            for other in available:
                if other != feature:
                    original = test[other].values[0]
                    noise = np.random.normal(0, 0.3)
                    test[other] = original + noise
            
            try:
                pred = model.predict(test)[0]
                predictions.append(pred)
            except:
                continue
        
        # Calculate stability (how often prediction stays the same)
        if predictions:
            stability = sum(p == original_pred for p in predictions) / len(predictions)
            value = user_features[feature].values[0]
            feature_stability.append((feature, stability, value))
    
    # Sort by stability
    feature_stability.sort(key=lambda x: x[1], reverse=True)
    
    # Build anchor rule from stable features
    conditions = []
    for feature, stability, value in feature_stability[:3]:  # Top 3 features
        if stability >= threshold:
            if feature == 'log_playtime_at_review':
                conditions.append(f"playtime > {np.expm1(value):.0f} hours")
            elif feature == 'sentiment_polarity':
                conditions.append(f"sentiment > {value:.2f}")
            elif feature == 'word_count':
                conditions.append(f"review_length > {int(value)} words")
            else:
                conditions.append(f"{feature} > {value:.0f}")
    
    if conditions:
        rule = "IF " + " AND ".join(conditions)
        rule += f" THEN {'RECOMMENDED' if original_pred else 'NOT RECOMMENDED'}"
        
        # Estimate precision
        test_preds = []
        for _ in range(100):
            test = user_features.copy()
            for f in available:
                if f not in [feat for feat, _, _ in feature_stability[:3]]:
                    test[f] = test[f].values[0] + np.random.normal(0, 0.5)
            try:
                test_preds.append(model.predict(test)[0])
            except:
                continue
        
        precision = sum(p == original_pred for p in test_preds) / len(test_preds) if test_preds else 0
        
        logger.info(f"Anchor found with precision {precision:.1%}")
        
        return {
            'anchor_rule': rule,
            'precision': float(precision),
            'num_conditions': len(conditions),
            'method': 'anchors'
        }
    else:
        logger.warning(f"No features meet threshold {threshold:.1%}")
        return {
            'anchor_rule': 'No stable conditions found',
            'precision': 0.0,
            'num_conditions': 0,
            'method': 'anchors'
        }

In [55]:
# Compare Three Counterfactual Methods
print("="*60)
print("COUNTERFACTUAL METHODS COMPARISON")
print("="*60)

# Test user (negative prediction expected)
test_user = pd.DataFrame({
    'user_games_owned': [15],
    'user_num_reviews': [3],
    'log_playtime_at_review': [2.5],  # ~11 hours
    'game_price': [0],
    'game_genre': ['Action,Adventure,Massively Multiplayer,Free To Play'],
    'game_description': ['PUBG: BATTLEGROUNDS, the high-stakes winner-take-all shooter...'],
    'review_text': ['Not great, too many bugs'],
    'sentiment_polarity': [-0.3],
    'word_count': [5]
})

# Get prediction
pred = pipeline_svm.predict(test_user)[0]
prob = pipeline_svm.predict_proba(test_user)[0][1]

print(f"\nUser: {test_user['review_text'].values[0]}")
print(f"Prediction: {'RECOMMENDED' if pred else 'NOT RECOMMENDED'} ({prob:.1%})")
print(f"Playtime: {np.expm1(test_user['log_playtime_at_review'].values[0]):.0f} hours")
print(f"Sentiment: {test_user['sentiment_polarity'].values[0]:.2f}")

# Method 1: Greedy
print("\n" + "-"*60)
print("METHOD 1: Greedy (single-feature changes)")
print("-"*60)
greedy = generate_counterfactuals(pipeline_svm, test_user, test_user.columns.tolist())
for i, cf in enumerate(greedy, 1):
    print(f"{i}. {cf['change']}")

# Method 2: DICE
print("\n" + "-"*60)
print("METHOD 2: DICE (multi-feature changes)")
print("-"*60)
dice = generate_counterfactuals_dice(pipeline_svm, test_user, test_user.columns.tolist())
for i, cf in enumerate(dice, 1):
    print(f"{i}. {cf['explanation']} (distance: {cf['distance']:.2f})")

# Method 3: Anchors (test on positive example)
print("\n" + "-"*60)
print("METHOD 3: Anchors (decision rules)")
print("-"*60)
positive_user = pd.DataFrame({
    'user_games_owned': [100], 'user_num_reviews': [25],
    'log_playtime_at_review': [6.0], 'game_price': [0],
    'game_genre': ['Action,Adventure,Massively Multiplayer,Free To Play'],
    'game_description': ['PUBG: BATTLEGROUNDS, the high-stakes winner-take-all shooter...'],
    'review_text': ['Amazing game!'], 'sentiment_polarity': [0.85], 'word_count': [8]
})
anchor = generate_anchors_explanation(pipeline_svm, positive_user, positive_user.columns.tolist())
print(f"Rule: {anchor['anchor_rule']}")
print(f"Precision: {anchor['precision']:.1%}")

print("\n" + "="*60)

COUNTERFACTUAL METHODS COMPARISON

User: Not great, too many bugs
Prediction: NOT RECOMMENDED (42.9%)
Playtime: 11 hours
Sentiment: -0.30

------------------------------------------------------------
METHOD 1: Greedy (single-feature changes)
------------------------------------------------------------
1. Improve sentiment from -0.30 to 0.20

------------------------------------------------------------
METHOD 2: DICE (multi-feature changes)
------------------------------------------------------------
INFO     | Generating DICE counterfactuals
INFO     | Generated 2 counterfactuals
1. sentiment: -0.30 → 0.00, review: 5 → 25 words (distance: 20.00)
2. playtime: 11h → 54h, sentiment: -0.30 → -0.10 (distance: 1.51)

------------------------------------------------------------
METHOD 3: Anchors (decision rules)
------------------------------------------------------------
INFO     | Generating anchor explanation
INFO     | Anchor found with precision 99.0%
Rule: IF user_games_owned > 100 AND 

---

## 5. Model Evaluation

Assess model performance using appropriate metrics.

In [ ]:
# TODO: Evaluate model on test set
# - Accuracy
# - Precision, Recall, F1-Score
# - ROC-AUC
# - Confusion Matrix

pass

In [ ]:
# TODO: Visualize results
# - Confusion matrix heatmap
# - ROC curve
# - Feature importance plot

pass

In [ ]:
# TODO: Print final evaluation summary

pass

---

## Summary

This notebook provides a complete pipeline for interpretable game recommendation predictions:

1. **Data Loading & Exploration** - Loaded and analyzed Steam review data
2. **Feature Engineering** - Extracted features from text and numeric data
3. **Model Training** - Trained binary classifier
4. **Prediction Function** - Generated predictions with counterfactual explanations
5. **Model Evaluation** - Assessed performance metrics

---

**Next Steps:**
- Implement remaining sections
- Tune model hyperparameters
- Improve counterfactual generation logic
- Add visualizations